# Module 1 Notebook: Jupyter and Ollama Orientation

**Purpose:** This notebook gives you a low-stakes introduction to Jupyter notebooks, Python cells, and the local Ollama model runtime used later in the course.

No prior programming experience is expected. Early notebooks are guided, but the course gradually moves from running instructor-provided templates to adapting and building AI-enabled code with AI assistance. You are expected to run the notebooks, inspect the evidence they produce, explain what happened in business terms, and take increasing responsibility for implementation choices.

**Use only fictional or instructor-provided data in this notebook. Do not enter confidential, personal, regulated, proprietary, or restricted information.**

## Learning Objectives

By the end of this notebook, you should be able to:

1. Run Markdown and Python cells in order.
2. Edit a small configuration cell without changing the rest of the notebook.
3. Recognize the difference between code, output, and written analysis.
4. Check whether Ollama is running on your computer.
5. Send a simple fictional prompt to a local model when one is available.
6. Save notebook evidence for later assignments.

## How to Use This Notebook

- Run cells from top to bottom.
- Edit only cells that say `TODO` or `Student configuration`.
- If something behaves strangely, use **Kernel -> Restart Kernel and Run All Cells**.
- Read error messages before asking for help; they usually identify the missing setup step.
- Save the notebook after a successful run so the outputs remain visible.

## Part 1: Your First Code Cell

A notebook mixes explanation, code, and output. This is a code cell. Run it and inspect the output below the cell.

In [ ]:
from datetime import datetime
from pathlib import Path

run_time = datetime.now().astimezone().isoformat(timespec="seconds")
print("Notebook is running.")
print(f"Run time: {run_time}")
print(f"Working folder: {Path.cwd()}")

## Part 2: Student Configuration

A configuration cell collects the values you are allowed to edit. Later notebooks use this pattern so you do not have to understand every line of helper code.

In [ ]:
# Student configuration
# TODO: Replace Your Name with your name.

STUDENT_NAME = "Your Name"
MODEL_NAME = "gemma3:1b"
OLLAMA_BASE_URL = "http://localhost:11434"

# Keep this False unless the instructor asks you to download the model from this notebook.
# A model pull can take several minutes and requires internet access.
AUTO_PULL_MODEL = False

print(f"Student: {STUDENT_NAME}")
print(f"Target local model: {MODEL_NAME}")
print(f"Ollama address: {OLLAMA_BASE_URL}")
print(f"Auto-pull model if missing: {AUTO_PULL_MODEL}")

## Part 3: A Tiny Python Example

Python variables store values. Later notebooks use variables for prompts, model names, settings, and output records.

In [ ]:
business_cases = [
    "Classify customer service emails by urgency.",
    "Summarize a meeting transcript for action items.",
    "Draft a response using an approved policy.",
]

for number, case in enumerate(business_cases, start=1):
    print(f"Case {number}: {case}")

## Part 4: What Ollama Does

Ollama is a local model runtime. It downloads model files, starts a local service, and lets notebooks send prompts to a model through `localhost`.

In later modules, Ollama lets the class run controlled model experiments without requiring paid API keys. Local does not mean risk-free: you should still avoid sensitive data because notebooks, outputs, downloaded models, and exported files can be shared or stored accidentally.

In [ ]:
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json


def ollama_request(path, payload=None, timeout=30):
    """Send a JSON request to the local Ollama server."""
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        OLLAMA_BASE_URL + path,
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    with urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def check_ollama():
    try:
        tags = ollama_request("/api/tags", timeout=5)
        models = sorted(model.get("name", "") for model in tags.get("models", []))
        return True, models, None
    except (HTTPError, URLError, TimeoutError, ConnectionRefusedError) as error:
        return False, [], str(error)


ollama_running, installed_models, ollama_error = check_ollama()

if ollama_running:
    print("Ollama is running.")
    print("Installed models:")
    for model in installed_models or ["No models installed yet."]:
        print(f"  - {model}")
else:
    print("Ollama is not reachable from this notebook.")
    print("Install Ollama from https://ollama.com/download, start the app, then rerun this cell.")
    print(f"Technical detail: {ollama_error}")

## Optional: Download the Model

If the target model is not installed and your instructor wants you to download it from the notebook, set `AUTO_PULL_MODEL = True` in the configuration cell and run the next cell. Otherwise, skip it and use the terminal command shown in the output.

In [ ]:
if not ollama_running:
    print("Skip model download until Ollama is running.")
elif any(model == MODEL_NAME for model in installed_models):
    print(f"{MODEL_NAME} is already installed.")
elif AUTO_PULL_MODEL:
    print(f"Pulling {MODEL_NAME}. This may take several minutes...")
    result = ollama_request("/api/pull", {"name": MODEL_NAME, "stream": False}, timeout=600)
    print(result)
    ollama_running, installed_models, ollama_error = check_ollama()
else:
    print(f"{MODEL_NAME} is not installed.")
    print(f"To install it outside the notebook, run: ollama pull {MODEL_NAME}")

## Part 5: Send a Simple Fictional Prompt

This cell runs only if Ollama is available and the model is installed. The prompt uses fictional, low-risk content.

In [ ]:
prompt = """
Classify this fictional business task as extraction, summarization, drafting, or classification.

Task: A facilities coordinator receives emails about office maintenance issues and wants AI
to label each request as routine, elevated, or urgent.

Answer in three bullets: classification, reason, human review needed.
""".strip()

if not ollama_running:
    print("Ollama is not running, so this prompt was not sent.")
elif not any(model == MODEL_NAME for model in installed_models):
    print(f"{MODEL_NAME} is not installed, so this prompt was not sent.")
else:
    response = ollama_request(
        "/api/generate",
        {
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.0, "num_predict": 180},
        },
        timeout=120,
    )
    print(response.get("response", ""))

## Part 6: What Evidence Should You Preserve?

For later labs, visible notebook output is evidence. Keep outputs that show:

- the model name and settings used;
- the prompt or source packet used;
- the model output;
- runtime or metadata when the notebook provides it;
- your own evaluation, not just the AI response.

Do not clear outputs before submitting unless the assignment specifically instructs you to remove private information.

In [ ]:
orientation_record = {
    "student": STUDENT_NAME,
    "run_time": run_time,
    "target_model": MODEL_NAME,
    "ollama_running": ollama_running,
    "installed_models": installed_models,
    "sensitive_data_used": False,
}

print(json.dumps(orientation_record, indent=2))

## Troubleshooting

| Symptom | Likely issue | What to try |
|---|---|---|
| `Connection refused` | Ollama is not running | Start the Ollama app or run `ollama serve` |
| Model is missing | The model has not been downloaded | Run `ollama pull gemma3:1b` |
| Cell keeps running | Model download or generation is slow | Wait, then ask for help if it exceeds the expected time |
| Output changed after rerun | LLM output can vary | Record settings and evaluate the run you actually received |
| Notebook state seems confused | Cells ran out of order | Restart kernel and run all cells from top to bottom |

## Final Checklist

- [ ] I entered my name in the configuration cell.
- [ ] I ran the notebook from top to bottom.
- [ ] I know whether Ollama is running on my computer.
- [ ] I know whether the target model is installed.
- [ ] I understand that later labs require visible notebook evidence.
- [ ] I did not enter sensitive or restricted information.